# MNIST Dual Playground: Autoencoder vs. Variational Autoencoder (VAE)

**Goal (≈ 40–50 min in pairs):**
- Train a **standard Autoencoder (AE)** to compress and reconstruct digits.
- Train a **VAE** to reconstruct *and* generate new digits.
- **Compare** the two side-by-side: reconstructions, latent space, and generative quality.

---

| | Autoencoder | VAE |
|---|---|---|
| Latent space | Deterministic point | Probabilistic distribution |
| Loss | Reconstruction only | Reconstruction + KL divergence |
| Can generate new samples? | ❌ (not reliably) | ✅ (sample from prior) |
| Latent space structure | Irregular / unconstrained | Regularised (≈ Gaussian) |

## 1 · Setup

Before we write any model code, we import the libraries we need and set a fixed random seed. Setting the seed ensures that every run produces the same results, making experiments reproducible. We also detect whether a GPU is available — training on GPU is much faster, but the code runs fine on CPU too.

In [ ]:
# Uncomment if running in Colab:
# !pip -q install torch torchvision --upgrade

import math, random, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| Device:', device)

## 2 · Experiment Parameters — change these and re-run

> Change **one at a time** and record what changes.

In [ ]:
# === LATENT SPACE ===
LATENT_DIM = 2       # try: 2, 8, 32
                     # 2  → easy to visualise; reconstructions weaker
                     # 32 → better reconstructions; harder to sample from

# === VAE REGULARISATION ===
BETA = 1.0           # β-VAE weight on KL term; try: 0.5, 1.0, 4.0
                     # higher β → more regularised latent space, worse recon

# === TRAINING ===
AE_LR      = 1e-3    # Autoencoder learning rate
VAE_LR     = 1e-3    # VAE learning rate
AE_EPOCHS  = 5
VAE_EPOCHS = 5
BATCH_SIZE = 128

# === DATA AUGMENTATION ===
AUG = False          # set True to add RandomAffine jitter

# ── DataLoaders ──────────────────────────────────────────────────────────────
tfs = []
if AUG:
    tfs.append(transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)))
tfs.append(transforms.ToTensor())
tf = transforms.Compose(tfs)

train_ds = datasets.MNIST('./data', train=True,  download=True, transform=tf)
test_ds  = datasets.MNIST('./data', train=False, download=True, transform=tf)

nw = 0 if sys.platform == 'win32' else 2
pin = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=nw, pin_memory=pin)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=nw, pin_memory=pin)

print(f"Train: {len(train_ds):,} | Test: {len(test_ds):,}")
print(f"LATENT_DIM={LATENT_DIM}  BETA={BETA}  AUG={AUG}")
print(f"AE:  lr={AE_LR}  epochs={AE_EPOCHS}")
print(f"VAE: lr={VAE_LR}  epochs={VAE_EPOCHS}")

## 3 · Model Architectures

Both models share the **same convolutional backbone** so any difference in results is purely due to the training objective, not capacity.

In [ ]:
# ── Shared building blocks ────────────────────────────────────────────────────
def make_encoder():
    """Conv encoder: 1x28x28 → flat 64*7*7 feature vector."""
    return nn.Sequential(
        nn.Conv2d(1, 32, 3, stride=2, padding=1),   # → 32×14×14
        nn.ReLU(inplace=True),
        nn.Conv2d(32, 64, 3, stride=2, padding=1),  # → 64×7×7
        nn.ReLU(inplace=True),
        nn.Conv2d(64, 64, 3, stride=1, padding=1),  # → 64×7×7
        nn.ReLU(inplace=True),
    )

def make_decoder():
    """Conv decoder: 64×7×7 feature map → 1×28×28 logits."""
    return nn.Sequential(
        nn.ConvTranspose2d(64, 64, 3, stride=1, padding=1),                    # → 64×7×7
        nn.ReLU(inplace=True),
        nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),  # → 32×14×14
        nn.ReLU(inplace=True),
        nn.ConvTranspose2d(32,  1, 3, stride=2, padding=1, output_padding=1),  # → 1×28×28
        # No sigmoid here — outputs are logits; sigmoid applied at display time
    )

ENC_FLAT = 64 * 7 * 7  # flattened encoder output size


# ── Standard Autoencoder ──────────────────────────────────────────────────────
class ConvAE(nn.Module):
    """
    Standard (deterministic) Convolutional Autoencoder.
    Encoder maps x → z (a single point in latent space).
    Decoder maps z → x̂.
    Trained with MSE reconstruction loss.
    """
    def __init__(self, latent_dim=2):
        super().__init__()
        self.enc_cnn = make_encoder()
        self.fc_enc  = nn.Linear(ENC_FLAT, latent_dim)   # compress to z
        self.fc_dec  = nn.Linear(latent_dim, ENC_FLAT)   # expand from z
        self.dec_cnn = make_decoder()

    def encode(self, x):
        h = self.enc_cnn(x).view(x.size(0), -1)
        return self.fc_enc(h)          # deterministic z

    def decode(self, z):
        h = self.fc_dec(z).view(z.size(0), 64, 7, 7)
        return self.dec_cnn(h)

    def forward(self, x):
        z      = self.encode(x)
        x_logits = self.decode(z)
        return x_logits, z


# ── Variational Autoencoder ───────────────────────────────────────────────────
class ConvVAE(nn.Module):
    """
    Convolutional Variational Autoencoder.
    Encoder maps x → (μ, log σ²), i.e. parameters of a Gaussian.
    Reparameterisation trick: z = μ + σ·ε,  ε ~ N(0,I).
    Loss = Recon (BCE) + β·KL(q(z|x) || p(z)).
    """
    def __init__(self, latent_dim=2):
        super().__init__()
        self.enc_cnn   = make_encoder()
        self.fc_mu     = nn.Linear(ENC_FLAT, latent_dim)  # mean of q(z|x)
        self.fc_logvar = nn.Linear(ENC_FLAT, latent_dim)  # log variance
        self.fc_dec    = nn.Linear(latent_dim, ENC_FLAT)
        self.dec_cnn   = make_decoder()

    def encode(self, x):
        h      = self.enc_cnn(x).view(x.size(0), -1)
        mu     = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        """Sample z using the reparameterisation trick (differentiable)."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)    # ε ~ N(0, I)
        return mu + eps * std          # z ~ N(μ, σ²)

    def decode(self, z):
        h = self.fc_dec(z).view(z.size(0), 64, 7, 7)
        return self.dec_cnn(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z          = self.reparameterize(mu, logvar)
        x_logits   = self.decode(z)
        return x_logits, mu, logvar


print("Models defined.")
print(f"  ConvAE  params: {sum(p.numel() for p in ConvAE(LATENT_DIM).parameters()):,}")
print(f"  ConvVAE params: {sum(p.numel() for p in ConvVAE(LATENT_DIM).parameters()):,}")

## 4 · Loss Functions

| Model | Loss | Formula |
|---|---|---|
| AE  | MSE recon | `mean((x̂ - x)²)` |
| VAE | BCE recon + β·KL | `BCE(logits, x) + β · KL(q‖p)` |

**Why different recon losses?**
- AE uses **MSE** (pixel regression, simple and effective for deterministic mapping).
- VAE uses **BCE-with-logits** (treats each pixel as a Bernoulli probability; consistent with the probabilistic generative model).

In [ ]:
def ae_loss(x_logits, x_true):
    """MSE between sigmoid(logits) and target pixels."""
    x_hat = torch.sigmoid(x_logits)
    return F.mse_loss(x_hat, x_true)


def vae_loss(x_logits, x_true, mu, logvar, beta=1.0):
    """
    ELBO loss = Reconstruction loss + β · KL divergence

    Recon: BCE with logits, summed over pixels, averaged over batch.
    KL:   -0.5 * Σ(1 + log σ² - μ² - σ²)  [closed form for diagonal Gaussian]
    """
    B     = x_true.size(0)
    recon = F.binary_cross_entropy_with_logits(x_logits, x_true, reduction='sum') / B
    kl    = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / B
    loss  = recon + beta * kl
    return loss, recon, kl

print("Loss functions defined.")

## 5 · Training Loops

A training loop iterates over every batch in the dataset, feeds it through the model, computes the loss, and updates the model weights via backpropagation. We write separate loops for the AE and VAE because they have different loss functions. The `@torch.no_grad()` decorator on the evaluation function tells PyTorch not to track gradients, saving memory and time when we are only measuring performance, not updating weights.

In [ ]:
# ── AE training ───────────────────────────────────────────────────────────────
def train_epoch_ae(model, loader, optimizer):
    model.train(); total_loss = total_n = 0
    for xb, _ in loader:
        xb = xb.to(device)
        x_logits, _ = model(xb)
        loss = ae_loss(x_logits, xb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_n    += xb.size(0)
    return total_loss / total_n

@torch.no_grad()
def eval_epoch_ae(model, loader):
    model.eval(); total_loss = total_n = 0
    for xb, _ in loader:
        xb = xb.to(device)
        x_logits, _ = model(xb)
        loss = ae_loss(x_logits, xb)
        total_loss += loss.item() * xb.size(0)
        total_n    += xb.size(0)
    return total_loss / total_n


# ── VAE training ──────────────────────────────────────────────────────────────
def train_epoch_vae(model, loader, optimizer, beta=1.0):
    model.train(); L = R = K = n = 0
    for xb, _ in loader:
        xb = xb.to(device)
        x_logits, mu, logvar = model(xb)
        loss, recon, kl = vae_loss(x_logits, xb, mu, logvar, beta)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        bs = xb.size(0)
        L += loss.item()*bs; R += recon.item()*bs; K += kl.item()*bs; n += bs
    return L/n, R/n, K/n

@torch.no_grad()
def eval_epoch_vae(model, loader, beta=1.0):
    model.eval(); L = R = K = n = 0
    for xb, _ in loader:
        xb = xb.to(device)
        x_logits, mu, logvar = model(xb)
        loss, recon, kl = vae_loss(x_logits, xb, mu, logvar, beta)
        bs = xb.size(0)
        L += loss.item()*bs; R += recon.item()*bs; K += kl.item()*bs; n += bs
    return L/n, R/n, K/n

print("Training functions defined.")

## 6 · Train the Autoencoder

We now instantiate the Autoencoder, create an Adam optimiser, and run the training loop for the number of epochs specified by `AE_EPOCHS`. After each epoch the **validation MSE** is printed — this tells us how well the model reconstructs digits it has never seen during training. A lower MSE means the reconstructed pixels are closer to the originals.

In [ ]:
ae = ConvAE(LATENT_DIM).to(device)
opt_ae = torch.optim.Adam(ae.parameters(), lr=AE_LR)

hist_ae = {"tr_loss": [], "va_loss": []}
print("Training Autoencoder...")
for ep in range(1, AE_EPOCHS + 1):
    trL = train_epoch_ae(ae, train_loader, opt_ae)
    vaL = eval_epoch_ae(ae, test_loader)
    hist_ae["tr_loss"].append(trL)
    hist_ae["va_loss"].append(vaL)
    print(f"[AE]  Epoch {ep:02d} | train MSE {trL:.5f} | val MSE {vaL:.5f}")

print("\n✅ AE training complete.")

## 7 · Train the VAE

We train the VAE in the same way, but notice that we now print **three** numbers per epoch: the total ELBO loss, the reconstruction term, and the KL divergence term. Early in training the KL is usually close to zero and reconstruction dominates. As training progresses the KL should rise gradually, indicating that the encoder is learning to produce a well-structured Gaussian latent space rather than collapsing to a point.

In [ ]:
vae = ConvVAE(LATENT_DIM).to(device)
opt_vae = torch.optim.Adam(vae.parameters(), lr=VAE_LR)

hist_vae = {"tr_loss": [], "tr_rec": [], "tr_kl": [],
            "va_loss": [], "va_rec": [], "va_kl": []}
print("Training VAE...")
for ep in range(1, VAE_EPOCHS + 1):
    trL, trR, trK = train_epoch_vae(vae, train_loader, opt_vae, BETA)
    vaL, vaR, vaK = eval_epoch_vae(vae, test_loader,  BETA)
    hist_vae["tr_loss"].append(trL); hist_vae["tr_rec"].append(trR); hist_vae["tr_kl"].append(trK)
    hist_vae["va_loss"].append(vaL); hist_vae["va_rec"].append(vaR); hist_vae["va_kl"].append(vaK)
    print(f"[VAE] Epoch {ep:02d} | train L {trL:.2f}  Recon {trR:.2f}  KL {trK:.2f} "
          f"| val L {vaL:.2f}  Recon {vaR:.2f}  KL {vaK:.2f}")

print("\n✅ VAE training complete.")

## 8 · Analysis

### 8.1 · Training Curves

With both models trained, we now compare them from four angles: *(1)* how quickly the loss decreases, *(2)* how sharp the reconstructions look, *(3)* how organised the latent space is, and *(4)* whether random samples from the latent space produce recognisable digits. Run each sub-section in order and fill in your observations in the Guided Tasks (Section 9).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# AE loss
ax = axes[0]
ax.plot(hist_ae["tr_loss"], 'b-o', label='AE train MSE')
ax.plot(hist_ae["va_loss"], 'b--s', label='AE val MSE')
ax.set_title('Autoencoder — MSE Loss'); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

# VAE loss — recon and KL on separate y-axes so scales don't crush each other
ax2 = axes[1]
ax2.plot(hist_vae["va_rec"], 'r-o', label='VAE val Recon (BCE)')
ax2.set_ylabel('Reconstruction loss', color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax3 = ax2.twinx()
ax3.plot(hist_vae["va_kl"], 'g--^', label='VAE val KL')
ax3.set_ylabel('KL divergence', color='g')
ax3.tick_params(axis='y', labelcolor='g')
ax2.set_title(f'VAE — Recon + KL (β={BETA})')
ax2.set_xlabel('Epoch')
lines1, lbl1 = ax2.get_legend_handles_labels()
lines2, lbl2 = ax3.get_legend_handles_labels()
ax2.legend(lines1 + lines2, lbl1 + lbl2, loc='upper right')
ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

### 8.2 · Side-by-Side Reconstructions

**What to look for:**
- AE reconstructions tend to be **sharper** (optimises directly for pixel similarity).
- VAE reconstructions are often slightly **blurrier** (trades sharpness for a structured latent space).

In [ ]:
@torch.no_grad()
def show_reconstructions_both(ae_model, vae_model, loader, n=8):
    ae_model.eval(); vae_model.eval()
    xb, _ = next(iter(loader))
    xb = xb[:n].to(device)

    ae_logits,  _      = ae_model(xb)
    vae_logits, _, _   = vae_model(xb)

    orig    = xb.cpu().numpy()
    ae_hat  = torch.sigmoid(ae_logits).cpu().numpy()
    vae_hat = torch.sigmoid(vae_logits).cpu().numpy()

    fig, axes = plt.subplots(3, n, figsize=(1.6*n, 5))
    row_labels = ['Original', 'AE recon', 'VAE recon']
    for row_idx, (images, label) in enumerate(zip([orig, ae_hat, vae_hat], row_labels)):
        for col in range(n):
            ax = axes[row_idx, col]
            ax.imshow(images[col, 0], cmap='gray', vmin=0, vmax=1)
            ax.axis('off')
            if col == 0:
                ax.set_ylabel(label, fontsize=10, rotation=0, labelpad=55, va='center')
    plt.suptitle('Reconstruction Comparison: AE vs VAE', fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()

show_reconstructions_both(ae, vae, test_loader, n=8)

### 8.3 · Latent Space Visualisation  *(works best with `LATENT_DIM = 2`)*

**What to look for:**
- **AE latent space** — points cluster per digit class but can be **spread irregularly**, with gaps. Sampling from a gap may produce garbage.
- **VAE latent space** — points are pushed towards `N(0,I)` by the KL term. Classes **overlap more** but the space is **continuous and gap-free** → sampling anywhere gives a recognisable digit.

In [ ]:
@torch.no_grad()
def plot_latent_space(ae_model, vae_model, loader, max_samples=5000):
    ae_model.eval(); vae_model.eval()
    ae_zs, vae_zs, labels = [], [], []
    n_seen = 0
    for xb, yb in loader:
        if n_seen >= max_samples: break
        xb = xb.to(device)
        ae_z  = ae_model.encode(xb)       # deterministic
        mu, _ = vae_model.encode(xb)      # use mean (no noise at eval)
        ae_zs.append(ae_z.cpu()); vae_zs.append(mu.cpu()); labels.append(yb)
        n_seen += xb.size(0)

    ae_zs  = torch.cat(ae_zs)[:max_samples].numpy()
    vae_zs = torch.cat(vae_zs)[:max_samples].numpy()
    labels = torch.cat(labels)[:max_samples].numpy()

    if ae_zs.shape[1] < 2:
        print("Set LATENT_DIM=2 to see a 2D scatter plot."); return

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    cmap = plt.cm.get_cmap('tab10', 10)
    for ax, zs, title in zip(axes, [ae_zs, vae_zs], ['AE Latent Space', 'VAE Latent Space']):
        sc = ax.scatter(zs[:, 0], zs[:, 1], c=labels, cmap=cmap, s=4, alpha=0.6, vmin=-0.5, vmax=9.5)
        plt.colorbar(sc, ax=ax, ticks=range(10), label='Digit')
        ax.set_title(title, fontsize=13); ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
        ax.grid(alpha=0.2)
    plt.suptitle(f'Latent Space (z={LATENT_DIM})', fontsize=14)
    plt.tight_layout(); plt.show()

plot_latent_space(ae, vae, test_loader)

### 8.4 · Generative Quality — Sampling from the Prior

**Key difference:**
- **AE**: We sample random z from `N(0,1)` — but the AE was *never* trained with any prior on z. Samples likely fall in gaps → blurry or meaningless outputs.
- **VAE**: The KL term explicitly pushes z towards `N(0,I)` during training. Sampling from `N(0,I)` at test time lands in known territory → recognisable digits.

In [ ]:
@torch.no_grad()
def show_samples_both(ae_model, vae_model, n=16):
    ae_model.eval(); vae_model.eval()
    z = torch.randn(n, LATENT_DIM, device=device)   # same z for fair comparison

    ae_x  = torch.sigmoid(ae_model.decode(z)).cpu().numpy()
    vae_x = torch.sigmoid(vae_model.decode(z)).cpu().numpy()

    rows_per_model = int(math.ceil(n ** 0.5))
    cols = rows_per_model

    fig, big_axes = plt.subplots(1, 2, figsize=(cols * 2.5, rows_per_model * 2.5 + 0.6))
    for big_ax, images, title in zip(big_axes, [ae_x, vae_x],
                                     ['AE (random z ~ N(0,I))', 'VAE (random z ~ N(0,I))']):
        big_ax.set_title(title, fontsize=12, pad=12)
        big_ax.axis('off')
        # Draw sub-grid manually
        for idx in range(n):
            r, c = divmod(idx, cols)
            ax = fig.add_axes([big_ax.get_position().x0 + c / cols * big_ax.get_position().width,
                               big_ax.get_position().y0 + (rows_per_model - 1 - r) / rows_per_model * big_ax.get_position().height,
                               big_ax.get_position().width  / cols,
                               big_ax.get_position().height / rows_per_model])
            ax.imshow(images[idx, 0], cmap='gray', vmin=0, vmax=1)
            ax.axis('off')

    plt.suptitle('Generated Samples: AE vs VAE', fontsize=14, y=1.01)
    plt.show()

show_samples_both(ae, vae, n=16)

### 8.5 · VAE Latent Space Traversal  *(requires `LATENT_DIM = 2`)*

Walk a regular grid across z₁ and z₂ and decode each point. This shows the **smooth, continuous** nature of the VAE latent space. The AE equivalent would show discontinuous jumps.

In [ ]:
@torch.no_grad()
def vae_latent_traversal(model, grid_size=15, z_range=3.0):
    if LATENT_DIM != 2:
        print("Set LATENT_DIM=2 to see the traversal."); return
    model.eval()
    grid = torch.linspace(-z_range, z_range, grid_size)
    canvas = np.zeros((28 * grid_size, 28 * grid_size))
    for i, z1 in enumerate(grid):
        for j, z2 in enumerate(grid):
            z = torch.tensor([[z1, z2]], device=device)
            img = torch.sigmoid(model.decode(z)).squeeze().cpu().numpy()
            canvas[(grid_size - 1 - i)*28:(grid_size - i)*28, j*28:(j+1)*28] = img

    plt.figure(figsize=(8, 8))
    plt.imshow(canvas, cmap='gray', vmin=0, vmax=1)
    plt.title(f'VAE Latent Space Traversal  (z₁, z₂ ∈ [{-z_range}, {z_range}])', fontsize=12)
    plt.xlabel('z₁ →'); plt.ylabel('← z₂')
    plt.xticks([]); plt.yticks([])
    plt.tight_layout(); plt.show()

vae_latent_traversal(vae, grid_size=15, z_range=3.0)

### 8.6 · Quantitative Comparison

Visual inspection is useful but subjective. Here we compute a single number — the **mean squared error (MSE)** between the original and reconstructed pixels — for both models on the test set. This gives us a fair, objective comparison of reconstruction quality. Keep in mind that a lower MSE does not automatically mean a better model: the VAE may score higher MSE but produce far better *generative* samples because of its structured latent space.

In [ ]:
@torch.no_grad()
def compute_mse(model, loader, is_vae=False):
    """Compute MSE of pixel reconstructions (comparable between AE and VAE)."""
    model.eval(); total = n = 0
    for xb, _ in loader:
        xb = xb.to(device)
        if is_vae:
            logits, _, _ = model(xb)
        else:
            logits, _    = model(xb)
        xhat  = torch.sigmoid(logits)
        total += F.mse_loss(xhat, xb, reduction='sum').item()
        n     += xb.numel()
    return total / n

ae_mse  = compute_mse(ae,  test_loader, is_vae=False)
vae_mse = compute_mse(vae, test_loader, is_vae=True)

print("\n" + "="*48)
print(f"  {'Model':<12} {'Test MSE (↓ better)':>20}")
print("-"*48)
print(f"  {'AE':<12} {ae_mse:>20.6f}")
print(f"  {'VAE':<12} {vae_mse:>20.6f}")
print("="*48)
print()
print("Note: AE typically wins on MSE — it optimises directly for it.")
print("VAE trades some recon quality for a structured, generative latent space.")

## 9 · Guided Tasks

> **How to work through this section:**
> 1. Read the task description carefully before changing anything.
> 2. Go to **Section 2 (Parameters)**, change the value indicated, and re-run **only** cells 6 & 7 to retrain.
> 3. Then re-run the relevant analysis plots in Section 8.
> 4. Come back here and fill in your observations.
> 5. Change **one parameter at a time** — reset others to default before moving to the next task.
>
> **Default settings to return to between tasks:** `LATENT_DIM=2`, `BETA=1.0`, `AE_LR=1e-3`, `VAE_LR=1e-3`, `AUG=False`


## Task A · Latent Dimension — How much information fits in z?

**Background:** The latent dimension `z` is the bottleneck of the model. A small `z` forces the model to compress aggressively — useful features survive, noise is discarded. A large `z` can store more detail but may memorise noise or lose the clean structure we want.

**What to do:**
1. Set `LATENT_DIM = 2` → retrain both models → run §8.1, §8.2, §8.3, §8.6 → record below.
2. Set `LATENT_DIM = 16` → retrain both models → run the same plots → record below.

**Record your numbers:**

| | LATENT_DIM = 2 | LATENT_DIM = 16 |
|---|---|---|
| AE val MSE | | |
| VAE val Recon loss | | |
| VAE val KL | | |

**Observation questions** *(answer in 1–2 sentences each)*:

- **A1.** With `z=2`, do the AE and VAE reconstructions look sharp or blurry? Which model handles the compression better? ____

- **A2.** After increasing to `z=16`, does reconstruction quality improve for both models equally, or does one benefit more? ____

- **A3.** Look at the latent space scatter plot (§8.3) for `z=2`. Can you visually identify clusters for each digit class? Are the clusters cleanly separated or overlapping? ____

- **A4.** What is the trade-off of using a very large latent dimension (e.g., `z=64`)? Why not just always use a large `z`? ____


## Task B · β (VAE Regularisation Strength) — Reconstruction vs. Structure

**Background:** The β parameter controls how strongly the VAE is forced to keep its latent space Gaussian. A small β relaxes this pressure — the model reconstructs well but the latent space can be messy. A large β enforces a tightly organised latent space but may sacrifice reconstruction quality.

**What to do:**
1. Keep `LATENT_DIM = 2`.
2. Set `BETA = 0.5` → retrain VAE only (cell 7) → run §8.1, §8.2, §8.3 → record below.
3. Set `BETA = 4.0` → retrain VAE → run same plots → record below.

**Record your numbers:**

| | β = 0.5 | β = 1.0 (default) | β = 4.0 |
|---|---|---|---|
| VAE val Recon loss | | | |
| VAE val KL | | | |
| Latent space: compact or spread? | | | |

**Observation questions:**

- **B1.** At `β=0.5`, does the VAE reconstruction look better or worse than at `β=1.0`? What does the KL value tell you? ____

- **B2.** At `β=4.0`, what happens to reconstruction quality? Look at the side-by-side reconstructions (§8.2) — do digits become harder to recognise? ____

- **B3.** Compare the latent space scatter plots (§8.3) for `β=0.5` vs `β=4.0`. How does the shape and spread of the clusters change? ____

- **B4.** In your own words: what does β control, and why is there a trade-off between reconstruction quality and latent space regularity? ____


## Task C · Sampling Quality — Can the model *generate* new digits?

**Background:** This is one of the most important differences between AE and VAE. The AE is never trained to handle a specific distribution of z — it just learns a mapping. The VAE explicitly trains its encoder to produce z values that follow a standard Gaussian, so sampling from that Gaussian at test time gives meaningful results.

**What to do:**
1. Set `LATENT_DIM = 2`, `BETA = 1.0` (defaults).
2. Run §8.4 (sampling plot) — look at both the AE and VAE generated images.
3. Repeat with `LATENT_DIM = 16` and observe whether generation improves.

**Observation questions:**

- **C1.** With `z=2`: do the AE samples look like recognisable digits? What do they look like instead? ____

- **C2.** With `z=2`: do the VAE samples look like recognisable digits? Are they sharp or blurry? ____

- **C3.** With `z=16`: does the quality of VAE samples improve compared to `z=2`? Does the AE improve as well? Explain the difference. ____

- **C4.** The AE scores a **lower MSE** than the VAE (§8.6), yet its samples are worse. How can a model with better reconstruction score produce worse samples? ____

- **C5.** Run the VAE latent space traversal (§8.5) with `z=2`. Describe what you see as z₁ and z₂ change — do digits morph smoothly or jump abruptly? What does this tell you about the VAE latent space? ____

## Task D · Latent Space Shape — What does the encoder learn?

**Background:** The structure of the latent space determines whether a model can be used as a generator. An AE may cluster digits well in latent space, but the clusters are irregularly placed with gaps between them. The VAE's KL term forces all clusters towards the origin and prevents gaps.

**What to do:**
1. Set `LATENT_DIM = 2`.
2. Run §8.3 (latent space scatter plot) for both models.
3. Visually compare the two plots.

**Observation questions:**

- **D1.** In the **AE** scatter plot: are the 10 digit classes clearly separated from one another? Estimate how far apart the clusters are from the origin. ____

- **D2.** In the **VAE** scatter plot: are the clusters more tightly grouped around (0, 0)? Do adjacent digit classes (e.g., 3 and 8) overlap more than in the AE? ____

- **D3.** In the AE plot, identify a region of empty space between two clusters. What would happen if you tried to decode a z value from that empty region? ____

- **D4.** Why is it important for the VAE latent space to be continuous (no gaps)? How does this connect to the goal of generating new images? ____

- **D5.** With a higher β (try `β=4.0`), does the VAE latent space become more or less compact? How does this relate to what the KL term is doing mathematically? ____


## Task E · Augmentation Effect

**Background:** Data augmentation artificially increases the variety of training examples by applying small random transformations (here: slight rotations and shifts). This can help models generalise better.

**What to do:**
1. Set `AUG = True`, keep all other parameters at default.
2. Retrain both models (cells 6 & 7).
3. Run §8.2 (reconstructions) and §8.6 (quantitative comparison).

**Observation questions:**

- **E1.** Does augmentation improve or worsen the **AE** val MSE compared to no augmentation? ____

- **E2.** Does augmentation improve or worsen the **VAE** val Recon loss? Does the KL change? ____

- **E3.** Look at the reconstructions (§8.2). Are the augmented-model outputs visually different — smoother, sharper, or similar? ____

- **E4.** In general, when would you expect augmentation to help most — when the model is overfitting or underfitting? How can you tell which case applies here? ____

## Summary Results Table

Fill this in after completing all tasks. Use the **default** run for the first two rows, then fill in your best configuration.

| Model | LATENT_DIM | BETA | AUG | Val MSE | Val Recon | Val KL | Sample Quality (1–5) | Latent Space Description |
|------:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---|
| AE (default) | 2 | — | No | | — | — | | |
| VAE (default) | 2 | 1.0 | No | | | | | |
| AE (best config) | | — | | | — | — | | |
| VAE (best config) | | | | | | | | |





##Wrap-up Questions

Answer each in **2–3 sentences**:

1. **Which model produced sharper reconstructions, and why does that make sense given the loss function each uses?** ____

2. **Which model produced better generative samples? What property of the VAE enables this?** ____

3. **The VAE has higher MSE than the AE, but is generally considered the more powerful model for generation tasks. How do you reconcile this?** ____

4. **If a medical imaging team wanted to compress MRI scans for storage (not generation), which model would you recommend and why?** ____

5. **In your own words, what is the role of the KL divergence term in the VAE loss? What would happen if you removed it entirely?** ____